In [1]:
# 02_data_analysis.ipynb - 数据分析（适配同级目录结构）
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import platform

# 1. 中文字体配置（优先执行）
def set_chinese_font():
    system = platform.system()
    try:
        if system == "Windows":
            plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei"]
        elif system == "Darwin":
            plt.rcParams["font.sans-serif"] = ["PingFang SC", "Arial Unicode MS"]
        else:
            plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]
        plt.rcParams["axes.unicode_minus"] = False
        print(f"✅ 中文字体配置完成（{system}系统）")
    except:
        plt.rcParams["font.sans-serif"] = ["Arial"]
        print("⚠️ 中文字体加载失败，使用默认字体")

set_chinese_font()

# 2. 路径定义（同级文件夹）
DATA_CLEAN = Path("data_clean")   # 同级的data_clean
OUTPUT = Path("output")           # 同级的output
OUTPUT_TABLES = OUTPUT / "tables"
OUTPUT_CHARTS = OUTPUT / "charts"

# 确保输出文件夹存在
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)
OUTPUT_CHARTS.mkdir(parents=True, exist_ok=True)

# 3. 读取清洗后的数据
def load_clean_data():
    combined_path = DATA_CLEAN / "us_trade_combined.csv"
    if not combined_path.exists():
        raise FileNotFoundError(f"❌ 未找到清洗数据，请先运行01_data_clean.ipynb\n路径：{combined_path.absolute()}")
    
    combined_df = pd.read_csv(combined_path)
    print(f"✅ 读取清洗数据: {combined_path.absolute()}")
    print(f"📊 数据形状: {combined_df.shape}")
    print(f"📅 年份范围: {combined_df['fiscal_year'].min()} - {combined_df['fiscal_year'].max()}")
    return combined_df

# 4. 统计分析与表格输出
def stats_analysis(df):
    # 年度汇总
    year_summary = df.pivot_table(
        index="fiscal_year",
        columns="trade_type",
        values="value_usd",
        aggfunc="sum",
        fill_value=0
    )
    # 计算贸易差额
    if {"import", "export"}.issubset(year_summary.columns):
        year_summary["trade_balance"] = year_summary["export"] - year_summary["import"]
    # 保存年度汇总
    year_summary.to_csv(OUTPUT_TABLES / "yearly_trade_summary.csv")
    print(f"\n📈 年度汇总表已保存: {OUTPUT_TABLES / 'yearly_trade_summary.csv'}")
    
    # 2024年Top15（若2024年数据不存在，取最新年份）
    target_year = 2024 if 2024 in df["fiscal_year"].unique() else df["fiscal_year"].max()
    top15_import = df[(df["fiscal_year"]==target_year) & (df["trade_type"]=="import")].nlargest(15, "value_usd")
    top15_export = df[(df["fiscal_year"]==target_year) & (df["trade_type"]=="export")].nlargest(15, "value_usd")
    # 保存Top15
    top15_import.to_csv(OUTPUT_TABLES / f"{target_year}_top15_import.csv", index=False)
    top15_export.to_csv(OUTPUT_TABLES / f"{target_year}_top15_export.csv", index=False)
    
    print(f"🏆 {target_year}年Top15表已保存至: {OUTPUT_TABLES.absolute()}")
    return year_summary, top15_import, top15_export, target_year

# 5. 可视化生成
def generate_charts(df, year_summary, top15_import, top15_export, target_year):
    # 5.1 年度趋势图
    plt.figure(figsize=(12, 6))
    sns.lineplot(data=year_summary[["import", "export"]], marker="o", linewidth=2)
    plt.title(f"美国农业年度进出口趋势（{df['fiscal_year'].min()}-{target_year}）", fontsize=14)
    plt.xlabel("财年", fontsize=12)
    plt.ylabel("金额（美元）", fontsize=12)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_CHARTS / "yearly_trade_trend.png", dpi=300)
    plt.close()
    
    # 5.2 进口Top15
    if len(top15_import) > 0:
        plt.figure(figsize=(12, 8))
        sns.barplot(data=top15_import, x="value_usd", y="Country", palette="Blues_r")
        plt.title(f"{target_year}年美国农业进口来源Top15", fontsize=14)
        plt.xlabel("金额（美元）", fontsize=12)
        plt.ylabel("国家/地区", fontsize=12)
        plt.tight_layout()
        plt.savefig(OUTPUT_CHARTS / f"{target_year}_top15_import.png", dpi=300)
        plt.close()
    
    # 5.3 出口Top15
    if len(top15_export) > 0:
        plt.figure(figsize=(12, 8))
        sns.barplot(data=top15_export, x="value_usd", y="Country", palette="Reds_r")
        plt.title(f"{target_year}年美国农业出口目的地Top15", fontsize=14)
        plt.xlabel("金额（美元）", fontsize=12)
        plt.ylabel("国家/地区", fontsize=12)
        plt.tight_layout()
        plt.savefig(OUTPUT_CHARTS / f"{target_year}_top15_export.png", dpi=300)
        plt.close()
        
    print(f"\n✅ 所有图表已保存至: {OUTPUT_CHARTS.absolute()}")
    # 在 generate_charts 函数末尾调用该函数
    generate_key_partners_chart(df, target_year)


# 5.4 核心伙伴趋势图（新增）
def generate_key_partners_chart(df, target_year):
    # 定义核心伙伴列表（匹配数据中的国家名称）
    key_partners = ["Mexico", "Canada", "China", "European Union-27"]
    # 筛选核心伙伴数据
    key_df = df[df["Country"].isin(key_partners)]
    
    if len(key_df) > 0:
        plt.figure(figsize=(14, 7))
        sns.lineplot(
            data=key_df,
            x="fiscal_year",
            y="value_usd",
            hue="Country",
            style="trade_type",
            marker="o",
            linewidth=2
        )
        plt.title(f"美国与核心伙伴农业贸易趋势（{df['fiscal_year'].min()}-{target_year}）", fontsize=14)
        plt.xlabel("财年", fontsize=12)
        plt.ylabel("金额（美元）", fontsize=12)
        plt.legend(loc="upper left")
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(OUTPUT_CHARTS / "key_partners_trend.png", dpi=300)
        plt.close()
        print(f"✅ 核心伙伴趋势图已保存: {OUTPUT_CHARTS / 'key_partners_trend.png'}")
    else:
        print("⚠️ 核心伙伴数据不足，跳过趋势图生成")




# 6. 主执行流程
if __name__ == "__main__":
    try:
        # 读取数据
        us_trade_df = load_clean_data()
        
        # 统计分析
        year_summary, top15_import, top15_export, target_year = stats_analysis(us_trade_df)
        
        # 生成可视化
        generate_charts(us_trade_df, year_summary, top15_import, top15_export, target_year)
        
        print(f"\n🎉 数据分析完成！结果已保存至 {OUTPUT.absolute()}")
        
    except Exception as e:
        print(f"\n❌ 错误: {str(e)}")

✅ 中文字体配置完成（Windows系统）
✅ 读取清洗数据: c:\Users\wujunyi\Desktop\小谢旁听\2026春-数据分析概论\homework\ex_team6-group\data_clean\us_trade_combined.csv
📊 数据形状: (1050, 4)
📅 年份范围: 1990 - 2024

📈 年度汇总表已保存: output\tables\yearly_trade_summary.csv
🏆 2024年Top15表已保存至: c:\Users\wujunyi\Desktop\小谢旁听\2026春-数据分析概论\homework\ex_team6-group\output\tables


C:\Users\wujunyi\AppData\Local\Temp\ipykernel_15496\1533470026.py:92: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=top15_import, x="value_usd", y="Country", palette="Blues_r")
C:\Users\wujunyi\AppData\Local\Temp\ipykernel_15496\1533470026.py:103: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=top15_export, x="value_usd", y="Country", palette="Reds_r")



✅ 所有图表已保存至: c:\Users\wujunyi\Desktop\小谢旁听\2026春-数据分析概论\homework\ex_team6-group\output\charts
✅ 核心伙伴趋势图已保存: output\charts\key_partners_trend.png

🎉 数据分析完成！结果已保存至 c:\Users\wujunyi\Desktop\小谢旁听\2026春-数据分析概论\homework\ex_team6-group\output
